# Smart Receipt Assistant - Quick Start Guide

This notebook demonstrates how to use the Smart Receipt Assistant for receipt processing.

## Overview

1. Setup and Configuration
2. Using Chains for Receipt Processing
3. Using LangChain Tools
4. Using LangChain Agent
5. Batch Processing

## 1. Setup and Configuration

In [ ]:
# Install dependencies if needed
# !pip install -r requirements.txt

In [ ]:
# Import required modules
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Verify API keys are set
print("API Keys Status:")
print(f"  PADDLEOCR_ACCESS_TOKEN: {'✓' if os.getenv('PADDLEOCR_ACCESS_TOKEN') else '✗'}")
print(f"  AISTUDIO_API_KEY: {'✓' if os.getenv('AISTUDIO_API_KEY') else '✗'}")

In [ ]:
# Find sample invoices
sample_dir = Path("../sample_invoices")
sample_files = list(sample_dir.glob("*.pdf")) + list(sample_dir.glob("*.jpg")) + list(sample_dir.glob("*.png"))

print(f"Found {len(sample_files)} sample files:")
for f in sample_files:
    print(f"  - {f.name}")

## 2. Using Chains for Receipt Processing

In [ ]:
from src.chains.ocr_chain import OCRChain
from src.chains.extraction_chain import ExtractionChain
from src.chains.classification_chain import ClassificationChain

In [ ]:
# Initialize chains
ocr_chain = OCRChain()
extraction_chain = ExtractionChain()
classification_chain = ClassificationChain()

print("Chains initialized successfully!")

In [ ]:
# Process a sample receipt
if sample_files:
    sample_file = str(sample_files[0])
    print(f"Processing: {sample_file}")
    
    # Step 1: OCR
    print("\n[1] Running OCR...")
    ocr_result = ocr_chain.process(sample_file, enable_seal=True)
    print(f"    Text length: {len(ocr_result['text'])} chars")
    print(f"    Seals found: {len(ocr_result['seals'])}")
    
    # Step 2: Classification
    print("\n[2] Classifying...")
    classify_result = classification_chain.classify(ocr_result["text"])
    print(f"    Type: {classify_result['receipt_type']}")
    print(f"    Category: {classify_result['reimbursement_category']}")
    
    # Step 3: Extraction
    print("\n[3] Extracting...")
    extract_result = extraction_chain.extract_with_seals(
        ocr_result["text"],
        ocr_result["seals"]
    )
    print(json.dumps(extract_result, ensure_ascii=False, indent=2))
else:
    print("No sample files found!")

## 3. Using LangChain Tools

In [ ]:
from src.tools import ReceiptOCRTool, ReceiptExtractionTool, ReceiptClassificationTool

In [ ]:
# Initialize tools
ocr_tool = ReceiptOCRTool()
extraction_tool = ReceiptExtractionTool()
classification_tool = ReceiptClassificationTool()

print("Tools initialized successfully!")

In [ ]:
# Use tools to process a receipt
if sample_files:
    sample_file = str(sample_files[0])
    
    # OCR
    ocr_result = ocr_tool.invoke({"file_path": sample_file, "enable_seal": True})
    
    # Classification
    classify_result = classification_tool.invoke({"ocr_text": ocr_result["text"]})
    
    # Extraction
    extract_result = extraction_tool.invoke({
        "ocr_text": ocr_result["text"],
        "seals": ocr_result["seals"]
    })
    
    print("Results:")
    print(json.dumps(extract_result, ensure_ascii=False, indent=2))
else:
    print("No sample files found!")

## 4. Using LangChain Agent

In [ ]:
from src.agents import create_receipt_agent

In [ ]:
# Create agent
agent = create_receipt_agent(verbose=True)
print("Agent created successfully!")

In [ ]:
# Use agent to process receipt
if sample_files:
    sample_file = str(sample_files[0])
    
    # Process with agent
    result = agent.process(sample_file)
    
    print("\nAgent Result:")
    print(json.dumps(result, ensure_ascii=False, indent=2, default=str))
else:
    print("No sample files found!")

In [ ]:
# Natural language query example
if sample_files:
    sample_file = str(sample_files[0])
    
    query = f"What is the total amount on this receipt? File: {sample_file}"
    result = agent.invoke(query)
    
    print("Response:")
    print(result.get("output", "No output"))

## 5. Batch Processing

In [ ]:
# Batch process multiple receipts
if len(sample_files) > 1:
    print(f"Processing {len(sample_files)} files...")
    
    results = ocr_chain.batch_process([str(f) for f in sample_files[:3]])
    
    for i, result in enumerate(results):
        print(f"\nFile {i+1}: {result.get('file_path', 'Unknown')}")
        print(f"  Status: {result.get('status', 'Unknown')}")
        if result.get('status') == 'success':
            print(f"  Text length: {len(result.get('text', ''))} chars")
else:
    print("Need multiple sample files for batch processing demo.")

## Summary

In this notebook, we covered:

1. **Chains**: Traditional processing pipeline (OCR → Extraction → Classification)
2. **Tools**: LangChain-compatible tools for modular processing
3. **Agent**: Intelligent agent that can decide which tools to use
4. **Batch Processing**: Processing multiple files at once

### Next Steps

- Try with your own receipt images
- Explore the seal recognition feature
- Integrate with your workflow using the tools or agent